# Ladder Event Boundary Evaluation on Colab

This notebook keeps the baseline sanity-check evaluation and runs the 4-level ladder experiment with Qwen3-VL.


In [ ]:
%cd /content
!ls

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

## Clone or Update Repository

If the repository already exists in Colab, this cell pulls the latest code. If it does not exist, it clones the repo.


In [ ]:
from getpass import getpass
import os

REPO_URL = "github.com/norewyx0205/vlm-event-boundary.git"
REPO_DIR = "/content/vlm-event-boundary"

if not os.path.exists(REPO_DIR):
    token = os.environ["GH_TOKEN"]
    if token:
        !git clone https://{token}@{REPO_URL} {REPO_DIR}
    else:
        !git clone https://{REPO_URL} {REPO_DIR}
else:
    print("Repository already exists; pulling latest changes...")
    %cd {REPO_DIR}
    !git pull


In [ ]:
%cd /content/vlm-event-boundary
!ls

## Install Dependencies

These packages are needed for Qwen video input, video generation, and result analysis.


In [ ]:
!pip install -U transformers accelerate qwen-vl-utils decord opencv-python imageio-ffmpeg

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.set_default_device("cuda")


## Configuration

Qwen3-VL is the default model for the ladder experiment. You can change the model string here if needed.


In [ ]:
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
RESULT_DIR = "/content/vlm-event-boundary/results"

BASELINE_ANNOTATION = "/content/vlm-event-boundary/baseline_boundary_videos/annotations.jsonl"
SYNTHETIC_ANNOTATION = "/content/vlm-event-boundary/synthetic_boundary_videos/annotations.jsonl"
LADDER_ROOT = "/content/vlm-event-boundary/data/ladder_v1"
DATASET_VERSION = "ladder_v1"

## Generate Baseline and Synthetic Reference Datasets

This regenerates the legacy simple baseline and the harder synthetic reference set. These are kept as reference points outside the 4-level ladder.

In [ ]:
!python generate_2d_boundary_videos.py --dataset all

## Baseline Sanity Check

This keeps the earlier simple baseline. It should verify that Qwen3 can solve the easy before/after task.


In [ ]:
from pathlib import Path

for name, annotation in [
    ("baseline", BASELINE_ANNOTATION),
    ("synthetic", SYNTHETIC_ANNOTATION),
]:
    path = Path(annotation)
    print(f"{name} annotation exists:", path.exists())
    if path.exists():
        print(f"{name} eval rows:", sum(1 for _ in open(path)))
        print(f"{name} videos:", len(list((path.parent / "videos").glob("*.mp4"))))
    else:
        print(f"{name} files not found; run the generation cell above.")

In [ ]:
!python scripts/run_eval.py \
  --annotation_path "$BASELINE_ANNOTATION" \
  --model_name "$MODEL_NAME" \
  --dataset_name baseline_qwen3_sanity_check \
  --output_dir "$RESULT_DIR"

## Synthetic Hard Reference Evaluation

This evaluates the legacy harder synthetic set so it can be compared with the simple baseline and the ladder levels.

In [ ]:
!python scripts/run_eval.py \
  --annotation_path "$SYNTHETIC_ANNOTATION" \
  --model_name "$MODEL_NAME" \
  --dataset_name synthetic_qwen3_reference \
  --output_dir "$RESULT_DIR"

## Generate 4-Level Ladder Dataset

This creates `data/ladder_v1/level_*` with evaluation-level mirrored annotations. Re-run this cell when generation parameters change.


In [ ]:
!python scripts/generate_ladder_dataset.py \
  --dataset_version "$DATASET_VERSION" \
  --samples_per_level 30 \
  --output_root "$LADDER_ROOT" \
  --seed 42


## Check Ladder Dataset

Each level should contain 30 base samples × 4 boundary conditions × 2 mirrored prompts = 240 evaluation rows.


In [ ]:
from pathlib import Path

for ann in sorted(Path(LADDER_ROOT).glob("level_*/annotations.jsonl")):
    video_count = len(list((ann.parent / "videos").glob("*.mp4")))
    row_count = sum(1 for _ in open(ann))
    print(ann.parent.name, "videos=", video_count, "eval_rows=", row_count)


## Qwen3 Ladder Smoke Test

Run a tiny subset before launching the full ladder evaluation.


In [ ]:
!python scripts/run_eval.py \
  --annotation_path "$LADDER_ROOT/level_1_simple/annotations.jsonl" \
  --model_name "$MODEL_NAME" \
  --dataset_name smoke_ladder_v1_level_1_simple_qwen3 \
  --output_dir "$RESULT_DIR" \
  --max_samples 4

## Run Qwen3 on All 4 Ladder Levels

This is the main ladder experiment. Results are saved under `results/<safe_model_name>/<dataset_name>/<timestamp>/`.


In [ ]:
LEVELS = [
    "level_1_simple",
    "level_2_randomized",
    "level_3_static_distractors",
    "level_4_moving_distractors",
]

for level_name in LEVELS:
    annotation_path = f"{LADDER_ROOT}/{level_name}/annotations.jsonl"
    dataset_name = f"{DATASET_VERSION}_{level_name}"
    print("Running", dataset_name)
    !python scripts/run_eval.py \
      --annotation_path "$annotation_path" \
      --model_name "$MODEL_NAME" \
      --dataset_name "$dataset_name" \
      --output_dir "$RESULT_DIR"

## Analyze Ladder Results

This aggregates all Qwen3 runs and computes accuracy tables plus swap-consistency diagnostics.


In [ ]:
ANALYSIS_DIR = f"analysis/{DATASET_VERSION}_ladder"

!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "ladder_v1_level_" \
  --output_dir "$ANALYSIS_DIR" \
  --plots

## Inspect Saved Files


In [ ]:
!find "$RESULT_DIR" -maxdepth 4 -type f | sort | tail -60
!find analysis -maxdepth 2 -type f | sort
